Considered the reference from  - https://medium.com/analytics-vidhya/multi-class-image-classification-using-alexnet-deep-learning-network-implemented-in-keras-api-c9ae7bc4c05f

In [97]:
from keras import layers
from keras.layers import Input, Dense, Activation,BatchNormalization, Flatten, Conv2D, MaxPooling2D,Conv2D,GlobalAveragePooling2D
from keras.models import Model
from keras.preprocessing import image
from keras.preprocessing.image import ImageDataGenerator
import keras.backend as K
K.set_image_data_format('channels_last')
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline
import os
import cv2
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report

In [98]:
# Define the paths to your image and csv folders
train_val_dir = "/content/drive/MyDrive/Colab Notebooks/charts/train_val"
test_dir = "/content/drive/MyDrive/Colab Notebooks/charts/test"
train_path_labels = "/content/drive/MyDrive/Colab Notebooks/charts/train_val.csv"
train_val_labels = pd.read_csv(train_path_labels)

In [99]:
# load training dataset in numpy array
images = []
labels = []
for filename in os.listdir(train_val_dir):
  if filename.endswith('.png'):
    # Load the images and resize them to (128, 128) with 3 color channels
    img = cv2.imread(os.path.join(train_val_dir, filename))
    img = cv2.resize(img, (128, 128))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # img = Image.open(os.path.join(train_val_dir, filename))
    img_array = np.array(img)
    # Append the array to the list of images
    images.append(img_array)
    labels.append(filename)
# Convert the string labels to numerical labels
le = LabelEncoder()
labels = le.fit_transform(labels)

# Convert the lists to NumPy arrays
images = np.array(images)
labels = np.array(labels)
# Save the arrays in NumPy format
np.save('x_train.npy', images)
np.save('y_train.npy', labels)
x_train = np.load('x_train.npy')
y_train = np.load('y_train.npy')

In [100]:
# load test dataset in numpy array
images = []
labels = []
for filename in os.listdir(test_dir):
  if filename.endswith('.png'):
    # Load the images and resize them to (128, 128) with 3 color channels
    img = cv2.imread(os.path.join(test_dir, filename))
    img = cv2.resize(img, (128, 128))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # img = Image.open(os.path.join(test_dir, filename))
    img_array = np.array(img)
    # Append the array to the list of images
    images.append(img_array)
    labels.append(filename)
# Convert the string labels to numerical labels
le = LabelEncoder()
labels = le.fit_transform(labels)
# Convert the lists to NumPy arrays

images = np.array(images)
labels = np.array(labels)
# Save the arrays in NumPy format
np.save('x_test.npy', images)
np.save('y_test.npy', labels)
x_test = np.load('x_test.npy')
y_test = np.load('y_test.npy')

# define some classes from the images we have observed
image_classes = ['line', 'dot_line', 'hbar_categorical', 'vbar_categorical', 'pie']
image_classes[0]
# map the categories to the labels array i.e y_train
label_map = {'line': 0, 'dot_line': 1, 'hbar_categorical': 2, 'vbar_categorical': 3, 'pie': 4}
y_train = np.array([label_map[label] for label in train_val_labels['type']])

In [101]:
print("Batch Size for Input Image : ",x_train.shape)
print("Batch Size for Output Image : ",x_train[0][1].shape)
print("Image Size of first image : ",x_train[0].shape)
print("Output of first image : ",x_train[0][1][0].shape)

Batch Size for Input Image :  (1000, 128, 128, 3)
Batch Size for Output Image :  (128, 3)
Image Size of first image :  (128, 128, 3)
Output of first image :  (3,)


In [102]:
def AlexNet(input_shape):
    
    X_input = Input(input_shape)
    # print(X_input)

    X = Conv2D(96,(11,11),strides = 4,name="conv0")(X_input)
    X = BatchNormalization(axis = 3 , name = "bn0")(X)
    X = Activation('relu')(X)

    X = MaxPooling2D((3,3),strides = 2,name = 'max0')(X)

    X = Conv2D(256,(5,5),padding = 'same' , name = 'conv1')(X)
    X = BatchNormalization(axis = 3 ,name='bn1')(X)
    X = Activation('relu')(X)

    X = MaxPooling2D((3,3),strides = 2,name = 'max1')(X)

    X = Conv2D(384, (3,3) , padding = 'same' , name='conv2')(X)
    X = BatchNormalization(axis = 3, name = 'bn2')(X)
    X = Activation('relu')(X)

    X = Conv2D(384, (3,3) , padding = 'same' , name='conv3')(X)
    X = BatchNormalization(axis = 3, name = 'bn3')(X)
    X = Activation('relu')(X)

    X = Conv2D(256, (3,3) , padding = 'same' , name='conv4')(X)
    X = BatchNormalization(axis = 3, name = 'bn4')(X)
    X = Activation('relu')(X)

    X = MaxPooling2D((3,3),strides = 2,name = 'max2')(X)

    X = Flatten()(X)

    X = Dense(4096, activation = 'relu', name = "fc0")(X)

    X = Dense(4096, activation = 'relu', name = 'fc1')(X) 

    X = Dense(6,activation='softmax',name = 'fc2')(X)

    model = Model(inputs = X_input, outputs = X, name='AlexNet')
    return model

In [103]:
alex = AlexNet(x_train.shape[1:])

In [104]:
alex.summary()

Model: "AlexNet"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_10 (InputLayer)       [(None, 128, 128, 3)]     0         
                                                                 
 conv0 (Conv2D)              (None, 30, 30, 96)        34944     
                                                                 
 bn0 (BatchNormalization)    (None, 30, 30, 96)        384       
                                                                 
 activation_15 (Activation)  (None, 30, 30, 96)        0         
                                                                 
 max0 (MaxPooling2D)         (None, 14, 14, 96)        0         
                                                                 
 conv1 (Conv2D)              (None, 14, 14, 256)       614656    
                                                                 
 bn1 (BatchNormalization)    (None, 14, 14, 256)       1024

In [105]:
alex.compile(optimizer = 'adam' , loss = 'categorical_crossentropy' , metrics=['accuracy'])

In [ ]:
alex.fit(x_train,y_train,epochs=50)

In [ ]:
preds = alex.evaluate_generator(x_test)
print ("Loss = " + str(preds[0]))
print ("Test Accuracy = " + str(preds[1]))

In [ ]:
predictions = alex.predict_generator(preds)